In [1]:
import os
# ─── THE ULTIMATE CHEAT CODE ───
# Tell Keras to use PyTorch instead of TensorFlow. Bypasses all Protobuf errors!
os.environ["KERAS_BACKEND"] = "torch" 

import requests
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout, BatchNormalization
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler

# Verify Keras is successfully using PyTorch
print(f"SUCCESS! Keras backend is now running on: {keras.backend.backend()}")

SUCCESS! Keras backend is now running on: torch


In [3]:
# ─── REGIONAL CONFIGURATION ───
CITIES = {
    'Jakarta':   {'lat': -6.2088, 'lon': 106.8456},
    'Bangkok':   {'lat': 13.7563, 'lon': 100.5018},
    'Manila':    {'lat': 14.5995, 'lon': 120.9842},
    'Singapore': {'lat': 1.3521,  'lon': 103.8198}
}

START_DATE = '2010-01-01'
END_DATE = '2024-12-31'
WINDOW_SIZE = 14
FEATURES = ['T_max', 'RH_max', 'Wind', 'Heat_Index', 'doy_sin', 'doy_cos']

def calculate_apparent_temperature(t, rh, ws):
    """Australian AT formula (incorporates wind cooling)"""
    e = (rh / 100.0) * 6.105 * np.exp((17.27 * t) / (237.7 + t))
    return t + (0.33 * e) - (0.70 * (ws / 3.6)) - 4.00

def create_sequences(data, target, window):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i : i + window])
        y.append(target[i + window])
    return np.array(X), np.array(y)

# ─── START THE REVENUE PIPELINE ───
for city, coords in CITIES.items():
    print(f"\n{'='*50}\n🛰️ PROTRACTED OPERATION: {city.upper()}\n{'='*50}")
    
    # 1. Fetch from Open-Meteo API
    url = (
        f"https://archive-api.open-meteo.com/v1/archive?latitude={coords['lat']}&longitude={coords['lon']}"
        f"&start_date={START_DATE}&end_date={END_DATE}"
        f"&daily=temperature_2m_max,relative_humidity_2m_max,windspeed_10m_max&timezone=Asia%2FSingapore"
    )
    response = requests.get(url).json()
    df = pd.DataFrame(response['daily'])
    df['time'] = pd.to_datetime(df['time'])
    df.set_index('time', inplace=True)
    df.columns = ['T_max', 'RH_max', 'Wind']
    df = df.ffill().bfill()
    
    # 💾 DATA STORING: Dump unmodified baseline down to raw folder
    df.to_csv(f'../data/raw/{city.lower()}_raw.csv')
    print(f"  -> Saved Raw: data/raw/{city.lower()}_raw.csv")

    # 2. Climate Math & Feature Engineering
    df['Heat_Index'] = calculate_apparent_temperature(df['T_max'], df['RH_max'], df['Wind'])
    df['doy_sin'] = np.sin(2 * np.pi * df.index.dayofyear / 365)
    df['doy_cos'] = np.cos(2 * np.pi * df.index.dayofyear / 365)
    
    # Calculate Localized 95th Percentile Limit
    threshold_95 = df['Heat_Index'].quantile(0.95)
    df['Target'] = (df['Heat_Index'] >= threshold_95).astype(int)
    
    # 💾 DATA STORING: Dump engineered features to processed folder
    df.to_csv(f'../data/processed/{city.lower()}_processed.csv')
    print(f"  -> Saved Processed: data/processed/{city.lower()}_processed.csv")

    # 3. Time Series Partitioning
    train_df = df[df.index < '2022-01-01']
    
    # Scale features relative strictly to the training split baseline
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train_df[FEATURES])
    
    # Save mathematical artifacts for Streamlit lookup
    joblib.dump(scaler, f'../models/scaler_{city.lower()}.pkl')
    joblib.dump(threshold_95, f'../models/threshold_{city.lower()}.pkl')

    X_train, y_train = create_sequences(train_scaled, train_df['Target'].values, WINDOW_SIZE)

    # 4. Neural Net Construction
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(WINDOW_SIZE, len(FEATURES))),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        BatchNormalization(),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001), 
        loss='binary_crossentropy', 
        metrics=[keras.metrics.AUC(name='auc')]
    )
    
    # Fire up training with the 15x penalty weight engine active
    early_stop = EarlyStopping(monitor='val_auc', patience=8, mode='max', restore_best_weights=True)
    
    print(f"  🧠 Training LSTM (Threshold: {threshold_95:.2f}°C)...")
    model.fit(
        X_train, y_train, validation_split=0.15, epochs=50, batch_size=32,
        class_weight={0: 1.0, 1: 15.0}, callbacks=[early_stop], verbose=0
    )
    
    # Save final model file
    model.save(f'../models/{city.lower()}_lstm.keras')
    print(f"  🎯 Completed! Weights saved to models/{city.lower()}_lstm.keras")

print("\n🎉 PIPELINE EXECUTION REVENUE COMPLETE! All data stored and models baked.")


🛰️ PROTRACTED OPERATION: JAKARTA
  -> Saved Raw: data/raw/jakarta_raw.csv
  -> Saved Processed: data/processed/jakarta_processed.csv
  🧠 Training LSTM (Threshold: 42.67°C)...


d:\Project\heatwave-predictor\venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


KeyboardInterrupt: 

In [ ]:
# Let's run a check on all locations for the 2025-2026 window
END_SAFE_DATE = '2026-05-10'  # Safe archive buffer date

print(f"📊 RUNNING OUT-OF-SAMPLE TEST VALIDATION (2025-01-01 -> {END_SAFE_DATE})\n")

for city in CITIES.keys():
    city_lbl = city.lower()
    
    # 1. Load Live Verification Data
    df_future = fetch_weather_data(CITIES[city]['lat'], CITIES[city]['lon'], '2025-01-01', END_SAFE_DATE)
    df_future['Heat_Index'] = calculate_apparent_temperature(df_future['T_max'], df_future['RH_max'], df_future['Wind'])
    df_future['doy_sin'] = np.sin(2 * np.pi * df_future.index.dayofyear / 365)
    df_future['doy_cos'] = np.cos(2 * np.pi * df_future.index.dayofyear / 365)
    
    # Load stored limits
    t_95 = joblib.load(f'../models/threshold_{city_lbl}.pkl')
    df_future['Target'] = (df_future['Heat_Index'] >= t_95).astype(int)
    
    # 2. Scale & Sequencify
    sc = joblib.load(f'../models/scaler_{city_lbl}.pkl')
    future_scaled = sc.transform(df_future[FEATURES])
    X_f, y_f = create_sequences(future_scaled, df_future['Target'].values, WINDOW_SIZE)
    
    # 3. Model Inference
    net = keras.models.load_model(f'../models/{city_lbl}_lstm.keras')
    preds = net.predict(X_f, verbose=0).flatten()
    
    res_df = pd.DataFrame({'Actual': y_f, 'AI_Risk': preds}, index=df_future.index[WINDOW_SIZE:])
    
    print(f"📈 {city.upper()} - Top predicted risk anomalies captured:")
    display(res_df.sort_values(by='AI_Risk', ascending=False).head(3).round(4))
    print("-" * 60)

✅ Scaler and Threshold saved to '../models/'
X_train shape: (4369, 14, 6)
